In [1]:
import pandas as pd
import numpy as np
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
import os
import sys
parent_dir = os.path.dirname(os.environ["GTE_DIR"].replace("Glaciation_time_estimator",""))
GTE_DIR=os.environ["GTE_DIR"]
sys.path.insert(0, parent_dir)
from glaciation_time_estimator.auxiliary_func.config_reader import read_config
from glaciation_time_estimator.auxiliary_func.chunking_data import ChunkLoader

KeyboardInterrupt: 

In [12]:
# config = read_config(
#     os.path.join(GTE_DIR,'configs/2021_tracking/01_01.yaml'))
analyze_year=True
year=2010
global global_rmse
# global_rmse = config["Global_sqrt_mse"]
ice_cont_crit_frac = 0.05
# classifiacation_palette = ['#e41a1c', '#377eb8', "#4daf4a"]
classifiacation_palette = ['#e41a1c', '#377eb8', "#4daf4a"]
config= read_config("/wolke_scratch/dnikolo/Glaciation_time_estimator/config_n2o.yaml")

In [11]:
chunk = ChunkLoader([year],config =config )


In [21]:

combined_cloud_df = chunk.cloud_chunk
glaciations_df = chunk.glac_chunk
glaciating_clouds = chunk.glac_cloud_chunk
combined_cloud_df=combined_cloud_df[~combined_cloud_df.is_large_pix_cloud]
combined_cloud_df = combined_cloud_df[(combined_cloud_df.avg_lat >30) | (combined_cloud_df.avg_lat<-30)]

glaciations_df=glaciations_df[~glaciations_df.is_large_pix_cloud]
glaciations_df = glaciations_df[(glaciations_df.avg_lat >30) | (glaciations_df.avg_lat<-30)]
glaciating_clouds=glaciating_clouds[~glaciating_clouds.is_large_pix_cloud]
glaciating_clouds = glaciating_clouds[(glaciating_clouds.avg_lat >30) | (glaciating_clouds.avg_lat<-30)]


In [14]:
era_5_data = xr.open_dataset("/wolke_scratch/dnikolo/ERA5_Data/Feb_2010/data_stream-oper_stepType-instant.nc")

In [15]:
era_5_data

<xarray.Dataset> Size: 3GB
Dimensions:     (valid_time: 672, latitude: 641, longitude: 641)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 5kB 2010-02-01 ... 2010-02-28T23:...
  * latitude    (latitude) float64 5kB 80.0 79.75 79.5 ... -79.5 -79.75 -80.0
  * longitude   (longitude) float64 5kB -80.0 -79.75 -79.5 ... 79.5 79.75 80.0
    expver      (valid_time) <U4 11kB ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 1GB ...
    sst         (valid_time, latitude, longitude) float32 1GB ...
    sp          (valid_time, latitude, longitude) float32 1GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-06-25T14:49 GRIB to CDM+CF via cfgrib-0.9.1...

In [16]:
t2m = era_5_data["t2m"] 
sst = era_5_data["sst"]

In [17]:
times = t2m.coords["valid_time"].values

In [30]:
glaciating_time_period = glaciating_clouds[((glaciating_clouds['track_start_time']+glaciating_clouds['track_length'])>times[0]) & ((glaciating_clouds['track_start_time']+glaciating_clouds['track_length'])<times[-1])]
clouds_5_days = combined_cloud_df[((combined_cloud_df['track_start_time']+combined_cloud_df['track_length'])>times[0]) & ((combined_cloud_df['track_start_time']+combined_cloud_df['track_length'])<times[-1])]

In [31]:
sc_5_days = glaciating_time_period[glaciating_time_period["Cloud type"]=="Stratocumulus"]

In [32]:
sc_5_days["mean_temp"]=(sc_5_days["min_temp"]+sc_5_days["max_temp"])/2

/tmp/ipykernel_50969/3875720917.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sc_5_days["mean_temp"]=(sc_5_days["min_temp"]+sc_5_days["max_temp"])/2


In [33]:
sc_5_days["mean_temp"]

14520    -9.0
14521    -9.0
14524    -9.0
14530    -9.0
14536    -9.0
         ... 
23098   -21.0
23499   -27.0
23542   -27.0
23649   -27.0
24026   -27.0
Name: mean_temp, Length: 206, dtype: float64

In [34]:
def round_quarter(x):
    return round(x * 4) / 4

In [36]:
# before the loop
# sc_5_days['temp_diff'] = pd.Series(, dtype='object')

sc_5_days['is_sea']    = True          # defaults, will flip to False if needed
for cloud_ind,cloud in sc_5_days.iterrows():
    # sc_5_days.loc[cloud_ind, 'temp_diff'] = np.empty(len(cloud.lat_hist))
    # sc_5_days.loc[cloud_ind,"is_sea"] = True
    t_diff = np.empty(len(cloud.lat_hist))
    for loc_ind,lat in enumerate(cloud.lat_hist):
        lon=cloud.lon_hist[loc_ind]
        time = cloud.track_start_time + pd.Timedelta(minutes=15)*loc_ind
        sst_val = sst.sel(valid_time=time.round('h'), latitude=round_quarter(lat), longitude=round_quarter(lon)).values
        if np.isnan(sst_val):
            sc_5_days.loc[cloud_ind,"is_sea"] = False
            break
        #     continue
        # TODO: Make it use t2m instaed of sst
        t2m_val = t2m.sel(valid_time=time.round('h'), latitude=round_quarter(lat), longitude=round_quarter(lon)).values
        t_diff[loc_ind] = t2m_val - (cloud["mean_temp"]+273.15)
    # print(t_diff.mean(),t_diff.std(),t_diff.min(),t_diff.max())    
    sc_5_days.loc[cloud_ind,"t_diff"] = t_diff.mean()
    

/tmp/ipykernel_50969/20028790.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sc_5_days['is_sea']    = True          # defaults, will flip to False if needed
/tmp/ipykernel_50969/20028790.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sc_5_days.loc[cloud_ind,"t_diff"] = t_diff.mean()


3.5e-323 0.0 3.5e-323 3.5e-323
18.533214518229183 0.9739946003505784 16.098596191406273 20.809594726562523
22.56504109700523 0.35590701610734576 21.981805419921898 23.031823730468773
18.618455200195335 0.7513063034716191 17.401177978515648 19.936853027343773
nan nan nan nan
21.25610504150393 0.11677432511365175 20.990838623046898 21.381158447265648
23.088769531250023 0.49825277022850667 22.302728271484398 23.859521484375023
22.346298653738863 0.40386682586177697 21.642663574218773 22.956445312500023
20.2726013183594 0.4335020943157822 19.603601074218773 21.183740234375023
17.956914813701946 0.44342524142629997 17.345788574218773 18.673242187500023
2.77e-322 0.0 2.5e-323 5.24e-322
20.037974294026714 0.647120203070612 18.113183593750023 20.868798828125023
0.0 0.0 0.0 0.0
15.298479652404808 0.836556050078483 13.836968994140648 16.328698730468773
16.605278110504173 0.5768798282692829 15.298364257812523 17.619165039062523
11.372068568638415 0.07707334424734325 11.271783447265648 11.50493774

In [37]:
sc_5_days['temp_diff']

KeyError: 'temp_diff'

In [ ]:
a = sst.sel(valid_time=times[0],latitude=26,longitude=0).values
if np.isnan(a):
    print("nan")

nan


In [39]:
sc_5_days.keys()

Index(['Cloud_ID', 'Time [m]', 'Magnitude', 'Glac_start_ind', 'Glac_peak_ind',
       'Linear', 'line_rmse', 'Rate_arr', 'Mean_glac_rate',
       'Glaciation time [h]', 'tracknumber', 'is_large_pix_cloud',
       'is_cot_valid_cloud', 'is_ctp_valid_cloud', 'is_liq', 'is_mix',
       'is_ice', 'max_water_frac', 'max_ice_fraction', 'avg_size[km]',
       'max_size[km]', 'min_size[km]', 'avg_size[px]', 'max_size[px]',
       'min_size[px]', 'track_start_time', 'track_length', 'avg_cot',
       'avg_ctp', 'avg_ctt', 'glaciation_start_time', 'glaciation_end_time',
       'avg_lat', 'avg_lon', 'start_ice_fraction', 'end_ice_fraction',
       'ice_frac_hist', 'cot_hist', 'cot_std_hist', 'cot_nan_frac_hist',
       'ctp_hist', 'ctp_std_hist', 'ctp_nan_frac_hist', 'ctt_hist',
       'ctt_std_hist', 'lat_hist', 'lon_hist', 'size_hist_km', 'min_temp',
       'max_temp', 'pole', 'Hemisphere', 'Lifetime [h]', 'Radius [km]',
       'Level', 'Optical Thickness', 'Cloud type', 'Season', 'mean_temp',
 

In [38]:
sc_5_days['t_diff']*sc_5_days["avg_cth"]

KeyError: 'avg_cth'

In [ ]:
times[1]

numpy.datetime64('2022-01-01T01:00:00.000000000')

In [ ]:
t2m.interp(valid_time=times[0], latitude=75.3, longitude=0).values


array(262.68083496)